# 05 — Final Dataset Consolidation and Audit

This notebook builds the final hourly meteorological dataset for the Recife automatic weather station used in the project.

The final study period is 2018–2023.

The originally proposed 2024 data were not included because the Recife INMET automatic station (WMO A301) was not found in the official 2024 historical package inspected during data acquisition.

This stage:

- loads the existing Recife dataset for 2018;
- extracts WMO A301 from the national INMET files for 2019–2023;
- standardizes column names and numeric values;
- validates station identity;
- combines all available years;
- reconstructs the complete hourly timeline;
- distinguishes missing timestamps from missing meteorological measurements;
- audits the final dataset without imputing missing values.

In [1]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

RAW_DIR = Path("../data/raw")

DATA_2018_PATH = Path(
    "../data/processed/DataFrameRecife.csv"
)

OUTPUT_PATH = Path(
    "../data/processed/recife_2018_2023_hourly.csv"
)

YEARS_FROM_RAW = [
    2019,
    2020,
    2021,
    2022,
    2023,
]

STATION_CODE = "A301"

CHUNK_SIZE = 200_000


# ============================================================
# 2. CANONICAL INMET COLUMN NAMES
# ============================================================

canonical_columns = [
    "Data",
    "HORA (UTC)",
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)",
    "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "TEMPERATURA DO PONTO DE ORVALHO (°C)",
    "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)",
    "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)",
    "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)",
    "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "VENTO, DIREÇÃO HORARIA (gr) (° (gr))",
    "VENTO, RAJADA MAXIMA (m/s)",
    "VENTO, VELOCIDADE HORARIA (m/s)",
]


metadata_columns = [
    "ESTACAO",
    "CODIGO (WMO)",
    "UF",
]


# ============================================================
# 3. FINAL PROJECT COLUMN NAMES
# ============================================================

column_mapping = {
    "Data": "date",
    "HORA (UTC)": "time_utc",

    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)":
        "precipitation_mm",

    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)":
        "pressure_station_mb",

    "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)":
        "pressure_max_prev_hour_mb",

    "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)":
        "pressure_min_prev_hour_mb",

    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)":
        "temperature_c",

    "TEMPERATURA DO PONTO DE ORVALHO (°C)":
        "dew_point_c",

    "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)":
        "temperature_max_prev_hour_c",

    "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)":
        "temperature_min_prev_hour_c",

    "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)":
        "dew_point_max_prev_hour_c",

    "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)":
        "dew_point_min_prev_hour_c",

    "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)":
        "humidity_max_prev_hour_pct",

    "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)":
        "humidity_min_prev_hour_pct",

    "UMIDADE RELATIVA DO AR, HORARIA (%)":
        "humidity_pct",

    "VENTO, DIREÇÃO HORARIA (gr) (° (gr))":
        "wind_direction_deg",

    "VENTO, RAJADA MAXIMA (m/s)":
        "wind_gust_ms",

    "VENTO, VELOCIDADE HORARIA (m/s)":
        "wind_speed_ms",
}


numeric_columns = [
    column
    for column in column_mapping.values()
    if column not in [
        "date",
        "time_utc",
    ]
]


# ============================================================
# 4. HEADER NORMALIZATION
# ============================================================

def normalize_column_name(name):

    name = str(name)

    name = (
        name
        .strip()
        .replace("\n", "")
    )

    name = unicodedata.normalize(
        "NFKD",
        name
    )

    name = "".join(
        character
        for character in name
        if not unicodedata.combining(character)
    )

    name = name.upper()

    name = re.sub(
        r"[^A-Z0-9]",
        "",
        name
    )

    return name


all_expected_columns = (
    canonical_columns
    + metadata_columns
)

normalized_to_canonical = {
    normalize_column_name(column): column
    for column in all_expected_columns
}


def build_rename_map(columns):

    rename_map = {}

    for original_column in columns:

        normalized = normalize_column_name(
            original_column
        )

        if normalized in normalized_to_canonical:

            rename_map[original_column] = (
                normalized_to_canonical[normalized]
            )

    return rename_map


# ============================================================
# 5. DETECT CSV FORMAT
# ============================================================

def detect_csv_format(file_path):

    encodings = [
        "utf-8",
        "utf-8-sig",
        "latin-1",
    ]

    separators = [
        ",",
        ";",
    ]

    station_key = normalize_column_name(
        "CODIGO (WMO)"
    )

    for encoding in encodings:

        for separator in separators:

            try:

                header = pd.read_csv(
                    file_path,
                    sep=separator,
                    encoding=encoding,
                    nrows=0,
                )

                normalized_header = {
                    normalize_column_name(column)
                    for column in header.columns
                }

                if station_key in normalized_header:

                    return encoding, separator

            except Exception:
                continue

    raise ValueError(
        f"Could not determine CSV format for "
        f"{file_path.name}"
    )


# ============================================================
# 6. NUMERIC CONVERSION
# ============================================================

def convert_numeric_columns(df):

    for column in numeric_columns:

        df[column] = (
            df[column]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
        )

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    return df


# ============================================================
# 7. LOAD 2018 DATA
# ============================================================

if not DATA_2018_PATH.exists():

    raise FileNotFoundError(
        f"2018 dataset not found: "
        f"{DATA_2018_PATH}"
    )


df_2018 = pd.read_csv(
    DATA_2018_PATH,
    sep=",",
    encoding="utf-8",
)


df_2018.columns = (
    df_2018.columns
    .str.strip()
    .str.replace("\n", "", regex=False)
)


rename_2018 = build_rename_map(
    df_2018.columns
)

df_2018 = df_2018.rename(
    columns=rename_2018
)


missing_2018_columns = [
    column
    for column in canonical_columns
    if column not in df_2018.columns
]

if missing_2018_columns:

    raise ValueError(
        "2018 dataset is missing columns: "
        f"{missing_2018_columns}"
    )


df_2018 = df_2018[
    canonical_columns
].copy()


df_2018 = df_2018.rename(
    columns=column_mapping
)


df_2018 = convert_numeric_columns(
    df_2018
)


df_2018["datetime_utc"] = pd.to_datetime(
    df_2018["date"].astype(str).str.strip()
    + " "
    + df_2018["time_utc"].astype(str).str.strip(),
    errors="coerce",
    utc=True,
)


if df_2018["datetime_utc"].isna().any():

    raise ValueError(
        "Invalid datetime values found in 2018."
    )


df_2018["source_year"] = 2018
df_2018["record_present"] = True


print(
    "2018 observations loaded:",
    len(df_2018)
)


# ============================================================
# 8. LOAD AND FILTER 2019–2023
# ============================================================

raw_year_datasets = []


for year in YEARS_FROM_RAW:

    file_path = RAW_DIR / f"{year}.csv"

    if not file_path.exists():

        raise FileNotFoundError(
            f"Missing raw file: "
            f"{file_path}"
        )


    print()
    print(
        f"Processing {year}..."
    )


    encoding, separator = detect_csv_format(
        file_path
    )


    header = pd.read_csv(
        file_path,
        sep=separator,
        encoding=encoding,
        nrows=0,
    )


    rename_map = build_rename_map(
        header.columns
    )


    columns_needed = (
        canonical_columns
        + metadata_columns
    )


    found_columns = set(
        rename_map.values()
    )


    missing_columns = [
        column
        for column in columns_needed
        if column not in found_columns
    ]


    if missing_columns:

        raise ValueError(
            f"{year}.csv is missing columns: "
            f"{missing_columns}"
        )


    usecols = list(
        rename_map.keys()
    )


    year_chunks = []


    reader = pd.read_csv(
        file_path,
        sep=separator,
        encoding=encoding,
        usecols=usecols,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    )


    for chunk in reader:

        chunk = chunk.rename(
            columns=rename_map
        )


        chunk["CODIGO (WMO)"] = (
            chunk["CODIGO (WMO)"]
            .astype(str)
            .str.strip()
            .str.upper()
        )


        recife_chunk = chunk[
            chunk["CODIGO (WMO)"]
            == STATION_CODE
        ].copy()


        if not recife_chunk.empty:

            year_chunks.append(
                recife_chunk
            )


    if not year_chunks:

        raise ValueError(
            f"No A301 observations found "
            f"for {year}."
        )


    df_year = pd.concat(
        year_chunks,
        ignore_index=True,
    )


    # --------------------------------------------------------
    # Station validation
    # --------------------------------------------------------

    station_names = sorted(
        df_year["ESTACAO"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    station_states = sorted(
        df_year["UF"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    station_codes = sorted(
        df_year["CODIGO (WMO)"]
        .dropna()
        .astype(str)
        .unique()
    )


    print(
        "Station:",
        station_names
    )

    print(
        "State:",
        station_states
    )

    print(
        "WMO:",
        station_codes
    )


    if station_codes != [STATION_CODE]:

        raise ValueError(
            f"Unexpected station code "
            f"in {year}: "
            f"{station_codes}"
        )


    # --------------------------------------------------------
    # Keep meteorological columns only
    # --------------------------------------------------------

    df_year = df_year[
        canonical_columns
    ].copy()


    df_year = df_year.rename(
        columns=column_mapping
    )


    df_year = convert_numeric_columns(
        df_year
    )


    df_year["datetime_utc"] = pd.to_datetime(
        df_year["date"].astype(str).str.strip()
        + " "
        + df_year["time_utc"].astype(str).str.strip(),
        errors="coerce",
        utc=True,
    )


    invalid_datetime = (
        df_year["datetime_utc"]
        .isna()
        .sum()
    )


    if invalid_datetime > 0:

        raise ValueError(
            f"{invalid_datetime} invalid "
            f"timestamps found in {year}."
        )


    df_year["source_year"] = year
    df_year["record_present"] = True


    print(
        "Observations:",
        len(df_year)
    )


    raw_year_datasets.append(
        df_year
    )


# ============================================================
# 9. COMBINE 2018–2023
# ============================================================

df_combined = pd.concat(
    [
        df_2018,
        *raw_year_datasets,
    ],
    ignore_index=True,
)


df_combined = (
    df_combined
    .sort_values("datetime_utc")
    .drop_duplicates(
        subset="datetime_utc",
        keep="first",
    )
    .reset_index(drop=True)
)


print()
print(
    "Combined observed records:",
    len(df_combined)
)


# ============================================================
# 10. BUILD COMPLETE HOURLY TIMELINE
# ============================================================

full_hourly_index = pd.date_range(
    start="2018-01-01 00:00",
    end="2023-12-31 23:00",
    freq="h",
    tz="UTC",
)


df_final = (
    df_combined
    .set_index("datetime_utc")
    .reindex(full_hourly_index)
)


df_final.index.name = (
    "datetime_utc"
)


df_final["record_present"] = (
    df_final["record_present"]
    .fillna(False)
    .astype(bool)
)


df_final["is_missing_hour"] = (
    ~df_final["record_present"]
)


df_final["date"] = (
    df_final.index
    .strftime("%Y-%m-%d")
)


df_final["time_utc"] = (
    df_final.index
    .strftime("%H:%M")
)


df_final["year"] = (
    df_final.index.year
)


# ============================================================
# 11. YEARLY TIMELINE AUDIT
# ============================================================

year_audit = []


for year in range(2018, 2024):

    year_data = df_final[
        df_final["year"] == year
    ]

    expected_hours = len(
        year_data
    )

    observed_hours = int(
        year_data["record_present"]
        .sum()
    )

    missing_hours = (
        expected_hours
        - observed_hours
    )

    precipitation_missing_on_observed_rows = int(
        year_data.loc[
            year_data["record_present"],
            "precipitation_mm",
        ]
        .isna()
        .sum()
    )


    year_audit.append(
        {
            "year": year,
            "expected_hours": expected_hours,
            "observed_records": observed_hours,
            "missing_timestamps": missing_hours,
            "precipitation_missing_on_observed_records":
                precipitation_missing_on_observed_rows,
        }
    )


year_audit_df = pd.DataFrame(
    year_audit
).set_index("year")


print()
print(
    "=== YEARLY TIMELINE AUDIT ==="
)

display(
    year_audit_df
)


# ============================================================
# 12. MISSING METEOROLOGICAL VALUES
# ============================================================

observed_records = df_final[
    df_final["record_present"]
]


missing_values = (
    observed_records[
        numeric_columns
    ]
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)


missing_values_df = pd.DataFrame(
    {
        "missing_values":
            missing_values,

        "percentage_of_observed_records":
            (
                missing_values
                / len(observed_records)
                * 100
            ),
    }
)


print()
print(
    "=== MISSING VALUES ON EXISTING RECORDS ==="
)

display(
    missing_values_df
)


# ============================================================
# 13. COMMON NUMERIC SENTINEL AUDIT
# ============================================================

sentinel_values = [
    -99999,
    -9999,
    9999,
    99999,
]


sentinel_results = []


for column in numeric_columns:

    for sentinel in sentinel_values:

        count = int(
            (
                observed_records[column]
                == sentinel
            )
            .sum()
        )

        if count > 0:

            sentinel_results.append(
                {
                    "column": column,
                    "sentinel": sentinel,
                    "count": count,
                }
            )


sentinel_df = pd.DataFrame(
    sentinel_results
)


print()
print(
    "=== SENTINEL VALUE AUDIT ==="
)

if sentinel_df.empty:

    print(
        "No common numeric sentinel "
        "values were detected."
    )

else:

    display(
        sentinel_df
    )


# ============================================================
# 14. BASIC PHYSICAL VALIDATION
# ============================================================

physical_checks = {
    "negative_precipitation":
        (
            df_final["precipitation_mm"] < 0
        ).sum(),

    "humidity_below_0":
        (
            df_final["humidity_pct"] < 0
        ).sum(),

    "humidity_above_100":
        (
            df_final["humidity_pct"] > 100
        ).sum(),

    "wind_direction_below_0":
        (
            df_final["wind_direction_deg"] < 0
        ).sum(),

    "wind_direction_above_360":
        (
            df_final["wind_direction_deg"] > 360
        ).sum(),

    "negative_wind_speed":
        (
            df_final["wind_speed_ms"] < 0
        ).sum(),
}


physical_checks_df = pd.DataFrame(
    physical_checks.items(),
    columns=[
        "check",
        "violations",
    ],
)


print()
print(
    "=== BASIC PHYSICAL VALIDATION ==="
)

display(
    physical_checks_df
)


# ============================================================
# 15. FINAL SUMMARY
# ============================================================

total_expected_hours = len(
    df_final
)

total_observed_records = int(
    df_final["record_present"]
    .sum()
)

total_missing_timestamps = int(
    df_final["is_missing_hour"]
    .sum()
)


print()
print(
    "=== FINAL DATASET SUMMARY ==="
)

print(
    "Study period:"
)

print(
    df_final.index.min(),
    "to",
    df_final.index.max(),
)

print()

print(
    "Expected hourly timestamps:",
    total_expected_hours
)

print(
    "Existing records:",
    total_observed_records
)

print(
    "Missing timestamps:",
    total_missing_timestamps
)

print(
    "Missing timestamp percentage:",
    f"{total_missing_timestamps / total_expected_hours * 100:.2f}%"
)

print()

print(
    "First timestamp:",
    df_final.index.min()
)

print(
    "Last timestamp:",
    df_final.index.max()
)


# ============================================================
# 16. SAVE FINAL HOURLY DATASET
# ============================================================

final_columns = [
    "date",
    "time_utc",
    "record_present",
    "is_missing_hour",

    *numeric_columns,
]


df_final[
    final_columns
].to_csv(
    OUTPUT_PATH,
    index=True,
)


print()
print(
    "Final dataset saved to:"
)

print(
    OUTPUT_PATH
)

2018 observations loaded: 8279

Processing 2019...
Station: ['RECIFE']
State: ['PE']
WMO: ['A301']
Observations: 8760

Processing 2020...
Station: ['RECIFE']
State: ['PE']
WMO: ['A301']
Observations: 8784

Processing 2021...
Station: ['RECIFE']
State: ['PE']
WMO: ['A301']
Observations: 8760

Processing 2022...
Station: ['RECIFE']
State: ['PE']
WMO: ['A301']
Observations: 8760

Processing 2023...
Station: ['RECIFE']
State: ['PE']
WMO: ['A301']
Observations: 8760

Combined observed records: 52103

=== YEARLY TIMELINE AUDIT ===


,expected_hours,observed_records,missing_timestamps,precipitation_missing_on_observed_records
year,,,,
2018,8760,8279,481,0
2019,8760,8760,0,5
2020,8784,8784,0,8
2021,8760,8760,0,1268
2022,8760,8760,0,8760
2023,8760,8760,0,8760



=== MISSING VALUES ON EXISTING RECORDS ===


,missing_values,percentage_of_observed_records
wind_gust_ms,30319,58.190507
wind_direction_deg,30284,58.123333
wind_speed_ms,30284,58.123333
dew_point_max_prev_hour_c,18856,36.189855
dew_point_min_prev_hour_c,18856,36.189855
humidity_max_prev_hour_pct,18855,36.187935
humidity_min_prev_hour_pct,18855,36.187935
dew_point_c,18851,36.180258
humidity_pct,18851,36.180258
precipitation_mm,18801,36.084295



=== SENTINEL VALUE AUDIT ===
No common numeric sentinel values were detected.

=== BASIC PHYSICAL VALIDATION ===


,check,violations
0,negative_precipitation,0
1,humidity_below_0,0
2,humidity_above_100,0
3,wind_direction_below_0,0
4,wind_direction_above_360,0
5,negative_wind_speed,0



=== FINAL DATASET SUMMARY ===
Study period:
2018-01-01 00:00:00+00:00 to 2023-12-31 23:00:00+00:00

Expected hourly timestamps: 52584
Existing records: 52103
Missing timestamps: 481
Missing timestamp percentage: 0.91%

First timestamp: 2018-01-01 00:00:00+00:00
Last timestamp: 2023-12-31 23:00:00+00:00

Final dataset saved to:
..\data\processed\recife_2018_2023_hourly.csv


In [2]:
# ============================================================
# RAW SOURCE DIAGNOSTIC — 2021, 2022 AND 2023
# ============================================================

diagnostic_years = [
    2021,
    2022,
    2023,
]

diagnostic_columns = [
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "VENTO, DIREÇÃO HORARIA (gr) (° (gr))",
    "VENTO, RAJADA MAXIMA (m/s)",
    "VENTO, VELOCIDADE HORARIA (m/s)",
]

diagnostic_results = []


for year in diagnostic_years:

    file_path = RAW_DIR / f"{year}.csv"

    print()
    print("=" * 60)
    print(f"RAW DATA DIAGNOSTIC — {year}")
    print("=" * 60)

    encoding, separator = detect_csv_format(
        file_path
    )

    header = pd.read_csv(
        file_path,
        sep=separator,
        encoding=encoding,
        nrows=0,
    )

    rename_map = build_rename_map(
        header.columns
    )

    required_for_test = [
        "Data",
        "HORA (UTC)",
        "CODIGO (WMO)",
        *diagnostic_columns,
    ]

    original_columns = [
        original
        for original, canonical
        in rename_map.items()
        if canonical in required_for_test
    ]

    recife_chunks = []

    reader = pd.read_csv(
        file_path,
        sep=separator,
        encoding=encoding,
        usecols=original_columns,
        chunksize=200_000,
        low_memory=False,
    )

    for chunk in reader:

        chunk = chunk.rename(
            columns=rename_map
        )

        chunk["CODIGO (WMO)"] = (
            chunk["CODIGO (WMO)"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        recife = chunk[
            chunk["CODIGO (WMO)"] == "A301"
        ].copy()

        if not recife.empty:
            recife_chunks.append(
                recife
            )


    df_raw_year = pd.concat(
        recife_chunks,
        ignore_index=True,
    )

    print(
        "A301 rows:",
        len(df_raw_year)
    )


    # --------------------------------------------------------
    # Count raw values before our numeric conversion
    # --------------------------------------------------------

    for column in diagnostic_columns:

        raw_series = (
            df_raw_year[column]
            .astype("string")
            .str.strip()
        )

        valid_raw = (
            raw_series.notna()
            &
            ~raw_series.str.lower().isin(
                [
                    "",
                    "nan",
                    "null",
                    "none",
                ]
            )
        )

        raw_available = int(
            valid_raw.sum()
        )

        diagnostic_results.append(
            {
                "year": year,
                "variable": column,
                "rows": len(df_raw_year),
                "raw_values_available":
                    raw_available,
                "raw_values_missing":
                    len(df_raw_year)
                    - raw_available,
                "availability_pct":
                    raw_available
                    / len(df_raw_year)
                    * 100,
            }
        )


    # --------------------------------------------------------
    # Show some ORIGINAL values
    # --------------------------------------------------------

    print()
    print("Raw precipitation examples:")

    examples = (
        df_raw_year[
            "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)"
        ]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(10)
        .tolist()
    )

    print(examples)


# ============================================================
# RESULT TABLE
# ============================================================

raw_diagnostic_df = pd.DataFrame(
    diagnostic_results
)

print()
print(
    "=== RAW SOURCE AVAILABILITY ==="
)

display(
    raw_diagnostic_df.pivot(
        index="variable",
        columns="year",
        values="availability_pct",
    ).round(2)
)


RAW DATA DIAGNOSTIC — 2021
A301 rows: 8760

Raw precipitation examples:
['0', '1,2', ',8', ',4', ',6', ',2', '2', '3,2', '6,6', '6']

RAW DATA DIAGNOSTIC — 2022
A301 rows: 8760

Raw precipitation examples:
[]

RAW DATA DIAGNOSTIC — 2023
A301 rows: 8760

Raw precipitation examples:
[]

=== RAW SOURCE AVAILABILITY ===


year,2021,2022,2023
variable,,,
"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",85.53,0.0,0.0
"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",86.20,0.0,0.0
"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",86.20,0.0,0.0
"UMIDADE RELATIVA DO AR, HORARIA (%)",84.95,0.0,0.0
"VENTO, DIREÇÃO HORARIA (gr) (° (gr))",86.19,0.0,0.0
"VENTO, RAJADA MAXIMA (m/s)",86.20,0.0,0.0
"VENTO, VELOCIDADE HORARIA (m/s)",86.19,0.0,0.0


In [4]:
# ============================================================
# 2021 AVAILABILITY PATTERN AUDIT
# ============================================================

core_variables = [
    "precipitation_mm",
    "temperature_c",
    "humidity_pct",
    "pressure_station_mb",
    "wind_speed_ms",
]

df_2021 = df_final[
    df_final.index.year == 2021
].copy()


# ------------------------------------------------------------
# 1. Identify hours with complete core meteorological data
# ------------------------------------------------------------

df_2021["core_data_available"] = (
    df_2021[core_variables]
    .notna()
    .all(axis=1)
)


print("=== 2021 CORE DATA AVAILABILITY ===")
print()

print(
    "Total hours:",
    len(df_2021)
)

print(
    "Hours with complete core data:",
    int(df_2021["core_data_available"].sum())
)

print(
    "Hours with incomplete core data:",
    int((~df_2021["core_data_available"]).sum())
)

print(
    "Complete-data percentage:",
    f"{df_2021['core_data_available'].mean() * 100:.2f}%"
)


# ------------------------------------------------------------
# 2. First and last complete observations
# ------------------------------------------------------------

valid_2021 = df_2021[
    df_2021["core_data_available"]
]

print()
print("First complete observation:")
print(valid_2021.index.min())

print()
print("Last complete observation:")
print(valid_2021.index.max())


# ------------------------------------------------------------
# 3. Detect continuous periods of missing core data
# ------------------------------------------------------------

missing_mask = (
    ~df_2021["core_data_available"]
)

group_id = (
    missing_mask
    != missing_mask.shift()
).cumsum()


missing_periods = []

for _, group in df_2021.groupby(group_id):

    if not group["core_data_available"].iloc[0]:

        start = group.index.min()
        end = group.index.max()

        hours = len(group)

        missing_periods.append(
            {
                "start": start,
                "end": end,
                "hours": hours,
                "days": hours / 24,
            }
        )


missing_periods_df = pd.DataFrame(
    missing_periods
).sort_values(
    "hours",
    ascending=False
)


print()
print("=== LONGEST MISSING PERIODS IN 2021 ===")

display(
    missing_periods_df.head(15)
)


# ------------------------------------------------------------
# 4. Monthly availability
# ------------------------------------------------------------

monthly_availability = (
    df_2021
    .assign(
        month=df_2021.index.month
    )
    .groupby("month")[
        "core_data_available"
    ]
    .agg(
        [
            "sum",
            "count",
            "mean",
        ]
    )
)


monthly_availability.columns = [
    "complete_hours",
    "total_hours",
    "availability_ratio",
]


monthly_availability[
    "availability_pct"
] = (
    monthly_availability[
        "availability_ratio"
    ]
    * 100
)


monthly_availability = (
    monthly_availability[
        [
            "complete_hours",
            "total_hours",
            "availability_pct",
        ]
    ]
)


print()
print("=== MONTHLY AVAILABILITY — 2021 ===")

display(
    monthly_availability.round(2)
)

=== 2021 CORE DATA AVAILABILITY ===

Total hours: 8760
Hours with complete core data: 7382
Hours with incomplete core data: 1378
Complete-data percentage: 84.27%

First complete observation:
2021-01-01 00:00:00+00:00

Last complete observation:
2021-11-12 06:00:00+00:00

=== LONGEST MISSING PERIODS IN 2021 ===


,start,end,hours,days
44,2021-11-12 07:00:00+00:00,2021-12-31 23:00:00+00:00,1193,49.708333
1,2021-04-17 02:00:00+00:00,2021-04-19 12:00:00+00:00,59,2.458333
14,2021-07-28 01:00:00+00:00,2021-07-28 12:00:00+00:00,12,0.500000
21,2021-08-06 01:00:00+00:00,2021-08-06 10:00:00+00:00,10,0.416667
12,2021-07-26 04:00:00+00:00,2021-07-26 11:00:00+00:00,8,0.333333
18,2021-08-04 05:00:00+00:00,2021-08-04 12:00:00+00:00,8,0.333333
13,2021-07-27 07:00:00+00:00,2021-07-27 13:00:00+00:00,7,0.291667
23,2021-08-15 06:00:00+00:00,2021-08-15 11:00:00+00:00,6,0.250000
20,2021-08-05 08:00:00+00:00,2021-08-05 13:00:00+00:00,6,0.250000
26,2021-08-20 06:00:00+00:00,2021-08-20 11:00:00+00:00,6,0.250000



=== MONTHLY AVAILABILITY — 2021 ===


,complete_hours,total_hours,availability_pct
month,,,
1,744,744,100.00
2,672,672,100.00
3,743,744,99.87
4,661,720,91.81
5,740,744,99.46
6,719,720,99.86
7,708,744,95.16
8,690,744,92.74
9,701,720,97.36


## Final Audit Conclusion

The final data audit identified important differences between timestamp availability and actual meteorological measurement availability.

The Recife automatic weather station (WMO A301) provides a usable meteorological series from 2018 until November 12, 2021 at 06:00 UTC.

The years 2019 and 2020 contain complete hourly timestamp coverage. The 2018 dataset contains known temporal gaps, which remain explicitly represented as missing observations.

During 2021, meteorological availability remains generally high until November 12. From November 12 at 07:00 UTC through the end of the year, a continuous period of 1,193 hours without complete core meteorological observations was identified.

Although timestamps are present for 2022 and 2023, the raw source contains no meteorological measurements for the main variables used by this project. These years are therefore excluded from model development.

The Recife A301 station was also not identified in the official INMET historical package inspected for 2024.

Therefore, the effective modeling period is defined as:

**2018-01-01 00:00 UTC to 2021-11-12 06:00 UTC**

Missing observations within this interval will not be imputed as zero. Samples affected by unavailable historical features or future prediction targets will be excluded during model dataset construction.